# Personalized Recipe Recommendation System

**Course:** CS-GY 6513 Big Data  
**Team:** Bias & Variance

This final implementation notebook demonstrates a personalized recipe recommendation pipeline on the Food.com Recipes and User Interactions dataset. It uses PySpark for large-scale preprocessing and analysis, Spark MLlib ALS for collaborative filtering, Parquet and HDFS-ready storage, model persistence, and streaming-style feedback events for a reliable live demo.


## Execution Modes

This notebook supports two execution modes:

1. **COLAB_DEMO mode** runs PySpark locally in Google Colab, stores outputs in the local runtime or a user-selected Drive path, and simulates Kafka-style feedback ingestion with an in-memory queue.
2. **CLUSTER_MODE** is optional and intended for a real Hadoop or Spark environment with HDFS storage and real Kafka Structured Streaming.

For the final live demo, keep `RUN_MODE="colab"` and `USE_REAL_KAFKA=False`.


In [ ]:
RUN_MODE = "colab"  # options: "colab", "cluster"
USE_REAL_KAFKA = False

LOCAL_DATA_DIR = "/content/foodcom"
LOCAL_OUTPUT_DIR = "/content/recipe_recommender_outputs"

HDFS_BASE_DIR = "hdfs:///user/bigdata/recipe_recommender"
HDFS_RAW_DIR = f"{HDFS_BASE_DIR}/raw"
HDFS_PROCESSED_DIR = f"{HDFS_BASE_DIR}/processed"
HDFS_MODEL_DIR = f"{HDFS_BASE_DIR}/models/als_model"
HDFS_OUTPUT_DIR = f"{HDFS_BASE_DIR}/outputs"

KAFKA_BOOTSTRAP_SERVERS = "localhost:9092"
KAFKA_TOPIC = "recipe_ratings"

assert RUN_MODE in {"colab", "cluster"}, "RUN_MODE must be either 'colab' or 'cluster'."

print("RUN_MODE:", RUN_MODE)
print("USE_REAL_KAFKA:", USE_REAL_KAFKA)


## Setup

The setup cell installs missing packages only when needed, imports the notebook dependencies, and detects whether the runtime is Google Colab. No secrets are hardcoded anywhere in the notebook.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

required_packages = ["pyspark", "kaggle", "pandas", "matplotlib", "numpy"]

for package_name in required_packages:
    if importlib.util.find_spec(package_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

IS_COLAB = "google.colab" in sys.modules

print("Google Colab detected:", IS_COLAB)
print("Local data directory:", LOCAL_DATA_DIR)
print("Local output directory:", LOCAL_OUTPUT_DIR)


## Dataset Download and Credential Handling

In Colab demo mode, the notebook first checks whether `RAW_recipes.csv` and `RAW_interactions.csv` already exist locally. If they do not, it looks for `KAGGLE_USERNAME` and `KAGGLE_KEY` in the environment. If those are not available, the notebook allows a Colab user to upload `kaggle.json` interactively. Secrets are never printed.

In cluster mode, the expected workflow is to stage the raw CSV files into `HDFS_RAW_DIR` before running the load cell.


In [ ]:
local_data_path = Path(LOCAL_DATA_DIR)
local_data_path.mkdir(parents=True, exist_ok=True)

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json_path = kaggle_dir / "kaggle.json"

required_dataset_files = ["RAW_recipes.csv", "RAW_interactions.csv"]


def dataset_files_present(base_path: Path) -> bool:
    return all((base_path / file_name).exists() for file_name in required_dataset_files)


def configure_kaggle_credentials() -> str:
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return "Using Kaggle credentials from environment variables."

    if kaggle_json_path.exists():
        os.chmod(kaggle_json_path, 0o600)
        return f"Using existing kaggle.json at {kaggle_json_path}."

    if IS_COLAB:
        from google.colab import files

        uploaded = files.upload()
        if "kaggle.json" in uploaded:
            kaggle_json_path.write_bytes(uploaded["kaggle.json"])
            os.chmod(kaggle_json_path, 0o600)
            return f"Uploaded kaggle.json to {kaggle_json_path}."

    return (
        "Kaggle credentials were not configured. Upload RAW_recipes.csv and "
        "RAW_interactions.csv manually to LOCAL_DATA_DIR if you do not want to use Kaggle download."
    )


def download_foodcom_dataset() -> None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "kaggle",
            "datasets",
            "download",
            "-d",
            "shuyangli94/food-com-recipes-and-user-interactions",
            "--unzip",
            "-p",
            str(local_data_path),
        ]
    )


if RUN_MODE == "cluster":
    recipes_input_path = f"{HDFS_RAW_DIR}/RAW_recipes.csv"
    interactions_input_path = f"{HDFS_RAW_DIR}/RAW_interactions.csv"
    print("Cluster mode selected. Ensure the raw CSV files are staged in HDFS before running the load cell.")
else:
    if dataset_files_present(local_data_path):
        print("Dataset files already exist locally. Skipping Kaggle download.")
    else:
        credential_status = configure_kaggle_credentials()
        print(credential_status)

        if not dataset_files_present(local_data_path):
            if (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")) or kaggle_json_path.exists():
                download_foodcom_dataset()
                print("Downloaded the Food.com dataset to:", local_data_path)
            else:
                raise FileNotFoundError(
                    "RAW_recipes.csv and RAW_interactions.csv were not found locally, and Kaggle credentials "
                    "were not available. Upload the CSV files to LOCAL_DATA_DIR or provide Kaggle access."
                )

    recipes_input_path = f"{LOCAL_DATA_DIR}/RAW_recipes.csv"
    interactions_input_path = f"{LOCAL_DATA_DIR}/RAW_interactions.csv"

print("Recipes input path:", recipes_input_path)
print("Interactions input path:", interactions_input_path)


## Spark Session

The Colab demo uses a local Spark session. The optional cluster mode can add the Kafka connector package only when real Kafka streaming is enabled, which avoids breaking the default Colab run.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    FloatType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

spark_builder = (
    SparkSession.builder
    .appName("PersonalizedRecipeRecommendationSystem")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "64")
)

if RUN_MODE == "colab":
    spark_builder = spark_builder.master("local[*]")

if RUN_MODE == "cluster" and USE_REAL_KAFKA:
    spark_builder = spark_builder.config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
    )

spark = spark_builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark application name:", spark.sparkContext.appName)
print("Spark master:", spark.sparkContext.master)


## Dataset Load and Validation

`RAW_recipes.csv` contains quoted commas and multiline text fields, so the CSV reader must enable multiline parsing with matching quote and escape settings. After loading, the notebook casts the core columns to stable numeric types, drops malformed rows, and validates the result with counts, schema inspection, and sample rows.


In [ ]:
recipes_df = spark.read.csv(
    recipes_input_path,
    header=True,
    inferSchema=True,
    multiLine=True,
    quote='"',
    escape='"'
)

interactions_df = spark.read.csv(
    interactions_input_path,
    header=True,
    inferSchema=True,
    multiLine=True,
    quote='"',
    escape='"'
)

raw_recipe_count = recipes_df.count()
raw_interaction_count = interactions_df.count()

recipes_df = recipes_df.withColumn("id", F.col("id").cast(IntegerType()))
recipes_df = recipes_df.withColumn("minutes", F.col("minutes").cast(IntegerType()))
recipes_df = recipes_df.dropna(subset=["id", "name"])

interactions_df = interactions_df.withColumn("user_id", F.col("user_id").cast(IntegerType()))             .withColumn("recipe_id", F.col("recipe_id").cast(IntegerType()))             .withColumn("rating", F.col("rating").cast(FloatType()))

interactions_df = interactions_df.dropna(subset=["user_id", "recipe_id", "rating"])

recipes_df = recipes_df.cache()
interactions_df = interactions_df.cache()

clean_recipe_count = recipes_df.count()
clean_interaction_count = interactions_df.count()

removed_recipe_rows = raw_recipe_count - clean_recipe_count
removed_interaction_rows = raw_interaction_count - clean_interaction_count

print("Recipe rows loaded:", f"{raw_recipe_count:,}")
print("Interaction rows loaded:", f"{raw_interaction_count:,}")
print("Malformed or incomplete recipe rows removed:", f"{removed_recipe_rows:,}")
print("Malformed or incomplete interaction rows removed:", f"{removed_interaction_rows:,}")

print("\nRecipes schema")
recipes_df.printSchema()

print("\nInteractions schema")
interactions_df.printSchema()

print("\nSample recipes")
recipes_df.select("id", "name", "minutes", "tags").show(5, truncate=100)

print("Sample interactions")
interactions_df.select("user_id", "recipe_id", "rating").show(5, truncate=False)


## Big Data EDA

This section uses Spark aggregations to profile the Food.com dataset at scale. Only compact aggregated outputs are converted to pandas for plotting.


In [ ]:
eda_ratings_df = interactions_df.filter((F.col("rating") >= 1) & (F.col("rating") <= 5)).cache()

num_recipes = clean_recipe_count
num_interactions = eda_ratings_df.count()
num_unique_users = eda_ratings_df.select("user_id").distinct().count()
num_unique_recipes_with_ratings = eda_ratings_df.select("recipe_id").distinct().count()
possible_matrix_cells = num_unique_users * num_unique_recipes_with_ratings
matrix_sparsity = 1 - (num_interactions / possible_matrix_cells) if possible_matrix_cells else float("nan")

print("Number of recipes:", f"{num_recipes:,}")
print("Number of interactions used for EDA:", f"{num_interactions:,}")
print("Number of unique users:", f"{num_unique_users:,}")
print("Number of unique recipes with ratings:", f"{num_unique_recipes_with_ratings:,}")
print("Potential user-recipe matrix cells:", f"{possible_matrix_cells:,}")
print("Observed matrix sparsity:", f"{matrix_sparsity:.6%}")

rating_distribution_pd = (
    eda_ratings_df.groupBy("rating")
    .count()
    .orderBy("rating")
    .toPandas()
)

top_recipes_pd = (
    eda_ratings_df.groupBy("recipe_id")
    .agg(
        F.count("*").alias("rating_count"),
        F.avg("rating").alias("avg_rating")
    )
    .join(
        recipes_df.select(F.col("id").alias("recipe_id"), "name"),
        on="recipe_id",
        how="left"
    )
    .orderBy(F.desc("rating_count"))
    .limit(10)
    .toPandas()
)

cook_time_distribution_pd = (
    recipes_df.withColumn(
        "cook_time_bucket",
        F.when(F.col("minutes") <= 30, "quick")
         .when(F.col("minutes") <= 90, "medium")
         .otherwise("long")
    )
    .groupBy("cook_time_bucket")
    .count()
    .toPandas()
)

cook_time_distribution_pd["cook_time_bucket"] = pd.Categorical(
    cook_time_distribution_pd["cook_time_bucket"],
    categories=["quick", "medium", "long"],
    ordered=True
)
cook_time_distribution_pd = cook_time_distribution_pd.sort_values("cook_time_bucket")

fig, axes = plt.subplots(1, 3, figsize=(22, 5))

axes[0].bar(rating_distribution_pd["rating"].astype(str), rating_distribution_pd["count"], color="#1f77b4")
axes[0].set_title("Rating Distribution")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")

axes[1].barh(top_recipes_pd["name"].fillna("Unknown"), top_recipes_pd["rating_count"], color="#ff7f0e")
axes[1].invert_yaxis()
axes[1].set_title("Top 10 Most Rated Recipes")
axes[1].set_xlabel("Number of Ratings")
axes[1].set_ylabel("Recipe")

axes[2].bar(cook_time_distribution_pd["cook_time_bucket"].astype(str), cook_time_distribution_pd["count"], color="#2ca02c")
axes[2].set_title("Cooking Time Bucket Distribution")
axes[2].set_xlabel("Cooking Time Bucket")
axes[2].set_ylabel("Recipe Count")

plt.tight_layout()
plt.show()

print("Top recipes by number of ratings")
display(top_recipes_pd)


## Feature Engineering

ALS primarily learns from `user_id`, `recipe_id`, and `rating`. The recipe metadata below is still important because it supports interpretation, filtering, and clean recommendation display during the demo.


In [ ]:
recipes_features = recipes_df.select(
    F.col("id").alias("recipe_id"),
    "name",
    "minutes",
    "tags",
    "nutrition"
).withColumn(
    "clean_tags",
    F.regexp_replace(F.col("tags"), r"[\[\]']", "")
).withColumn(
    "num_tags",
    F.size(F.split(F.col("clean_tags"), ","))
).withColumn(
    "cook_time_bucket",
    F.when(F.col("minutes") <= 30, "quick")
     .when(F.col("minutes") <= 90, "medium")
     .otherwise("long")
).withColumn(
    "has_nutrition",
    F.when(F.col("nutrition").isNotNull(), 1).otherwise(0)
)

ratings = interactions_df.filter((F.col("rating") >= 1) & (F.col("rating") <= 5))

ratings_with_features = ratings.join(recipes_features, on="recipe_id", how="left")

print("Sample joined ratings with recipe features")
ratings_with_features.select(
    "user_id",
    "recipe_id",
    "rating",
    "name",
    "minutes",
    "cook_time_bucket",
    "num_tags",
    "has_nutrition"
).show(5, truncate=False)


## Filtering for Collaborative Filtering

Sparse users and sparse recipes can make matrix factorization unstable and slower. The filtering step keeps only users with at least 5 ratings and recipes with at least 5 ratings, then caches the final training table.


In [ ]:
ratings_before_filter_count = ratings.count()

users_with_enough_ratings = (
    ratings.groupBy("user_id")
    .count()
    .filter(F.col("count") >= 5)
    .select("user_id")
)

recipes_with_enough_ratings = (
    ratings.groupBy("recipe_id")
    .count()
    .filter(F.col("count") >= 5)
    .select("recipe_id")
)

final_ratings = (
    ratings.join(users_with_enough_ratings, on="user_id", how="inner")
    .join(recipes_with_enough_ratings, on="recipe_id", how="inner")
    .select("user_id", "recipe_id", "rating")
    .cache()
)

filtered_ratings_count = final_ratings.count()
filtered_user_count = final_ratings.select("user_id").distinct().count()
filtered_recipe_count = final_ratings.select("recipe_id").distinct().count()

print("Row count before filtering:", f"{ratings_before_filter_count:,}")
print("Row count after filtering:", f"{filtered_ratings_count:,}")
print("Unique users after filtering:", f"{filtered_user_count:,}")
print("Unique recipes after filtering:", f"{filtered_recipe_count:,}")


## Storage Modes

This notebook supports two storage modes. In Colab demo mode, outputs are saved locally or to Google Drive. In cluster mode, processed Parquet files and trained models are saved to HDFS.


In [ ]:
if RUN_MODE == "cluster":
    processed_ratings_path = f"{HDFS_PROCESSED_DIR}/ratings"
    processed_recipes_path = f"{HDFS_PROCESSED_DIR}/recipes"
else:
    os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)
    processed_ratings_path = f"{LOCAL_OUTPUT_DIR}/ratings_parquet"
    processed_recipes_path = f"{LOCAL_OUTPUT_DIR}/recipes_parquet"

final_ratings.write.mode("overwrite").parquet(processed_ratings_path)
recipes_features.write.mode("overwrite").parquet(processed_recipes_path)

print("Saved processed ratings to:", processed_ratings_path)
print("Saved processed recipe features to:", processed_recipes_path)


## Train and Test Split

The filtered ratings dataset is split into training and test sets with a fixed seed for repeatability.


In [ ]:
train_df, test_df = final_ratings.randomSplit([0.8, 0.2], seed=42)

train_df = train_df.cache()
test_df = test_df.cache()

train_count = train_df.count()
test_count = test_df.count()

print("Training ratings:", f"{train_count:,}")
print("Test ratings:", f"{test_count:,}")


## ALS Model Training

The proposal called for 50-dimensional latent vectors, so the ALS model uses `rank=50` with explicit ratings, non-negative factors, and a fixed random seed.


In [ ]:
from pyspark.ml.recommendation import ALS

als_rank = 50

als = ALS(
    userCol="user_id",
    itemCol="recipe_id",
    ratingCol="rating",
    rank=als_rank,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop",
    nonnegative=True,
    seed=42
)

model = als.fit(train_df)

user_factor_count = model.userFactors.count()
item_factor_count = model.itemFactors.count()

print("ALS model trained successfully.")
print("Rank:", model.rank)
print("Number of user factors:", f"{user_factor_count:,}")
print("Number of item factors:", f"{item_factor_count:,}")


## Evaluation

RMSE and MAE measure how close predicted ratings are to actual held-out ratings. These metrics are stable for sparse explicit-feedback datasets like Food.com.


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test_df).dropna(subset=["prediction"]).cache()

rmse_evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

mae_evaluator = RegressionEvaluator(
    metricName="mae",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = rmse_evaluator.evaluate(predictions)
mae = mae_evaluator.evaluate(predictions)

print(f"RMSE on Test Set: {rmse:.4f}")
print(f"MAE on Test Set: {mae:.4f}")
print("Predictions available for evaluation:", f"{predictions.count():,}")

actual_vs_predicted_sample_pd = (
    predictions.join(
        recipes_features.select("recipe_id", "name"),
        on="recipe_id",
        how="left"
    )
    .select("user_id", "recipe_id", "name", "rating", "prediction")
    .orderBy(F.rand(seed=42))
    .limit(10)
    .toPandas()
)

actual_vs_predicted_sample_pd["prediction"] = actual_vs_predicted_sample_pd["prediction"].round(4)

print("Sample actual vs predicted ratings")
display(actual_vs_predicted_sample_pd)


## Recommendation Generation

This section generates top-10 recommendations for all users and then shows a qualitative example for one sample user. The recipe metadata is joined back in so the recommendations are readable during the demo.


In [ ]:
top_n = 10
user_recs = model.recommendForAllUsers(top_n).cache()

sample_user_id = (
    final_ratings.groupBy("user_id")
    .count()
    .orderBy(F.desc("count"))
    .first()["user_id"]
)

sample_user_history_pd = (
    final_ratings.filter(F.col("user_id") == sample_user_id)
    .join(recipes_features, on="recipe_id", how="left")
    .orderBy(F.desc("rating"), F.asc("minutes"))
    .select("user_id", "recipe_id", "name", "rating", "minutes", "cook_time_bucket", "num_tags")
    .limit(5)
    .toPandas()
)

sample_recs = (
    user_recs.filter(F.col("user_id") == sample_user_id)
    .select("user_id", F.explode("recommendations").alias("rec"))
    .select(
        "user_id",
        F.col("rec.recipe_id").alias("recipe_id"),
        F.col("rec.rating").alias("predicted_rating")
    )
)

sample_recs_named = (
    sample_recs.join(recipes_features, on="recipe_id", how="left")
    .select("user_id", "recipe_id", "name", "predicted_rating", "minutes", "cook_time_bucket", "num_tags")
    .orderBy(F.desc("predicted_rating"))
)

sample_recommendation_count = sample_recs_named.count()
topn_recommendations_generated = int(
    user_recs.select(F.sum(F.size("recommendations")).alias("total_recommendations")).first()["total_recommendations"]
)

sample_recs_named.show(top_n, truncate=False)

sample_recs_pd = sample_recs_named.toPandas()
sample_recs_pd["predicted_rating"] = sample_recs_pd["predicted_rating"].round(4)

print("Sample user history")
display(sample_user_history_pd)

print("Top-N recommendations for the sample user")
display(sample_recs_pd)


## Model Persistence

The trained ALS model is saved to either the local demo output directory or HDFS. In Colab mode, the notebook also writes a JSON summary of the evaluation metrics and a CSV export of the sample recommendations.


In [ ]:
if RUN_MODE == "cluster":
    model_save_path = HDFS_MODEL_DIR
else:
    model_save_path = f"{LOCAL_OUTPUT_DIR}/als_model"

model.write().overwrite().save(model_save_path)
print("Saved ALS model to:", model_save_path)

metrics = {
    "rmse": float(rmse),
    "mae": float(mae),
    "train_rows": int(train_count),
    "test_rows": int(test_count),
    "rank": int(als_rank)
}

metrics_save_path = None
sample_recommendations_save_path = None

if RUN_MODE == "cluster":
    metrics_save_path = f"{HDFS_OUTPUT_DIR}/metrics_json"
    spark.createDataFrame([metrics]).coalesce(1).write.mode("overwrite").json(metrics_save_path)
else:
    metrics_save_path = f"{LOCAL_OUTPUT_DIR}/metrics_summary.json"
    with open(metrics_save_path, "w", encoding="utf-8") as metrics_file:
        json.dump(metrics, metrics_file, indent=2)

    sample_recommendations_save_path = f"{LOCAL_OUTPUT_DIR}/sample_user_recommendations.csv"
    sample_recs_pd.to_csv(sample_recommendations_save_path, index=False)

print("Saved metrics to:", metrics_save_path)
if sample_recommendations_save_path:
    print("Saved sample recommendations to:", sample_recommendations_save_path)


## Streaming Event Ingestion

### A. Demo-Safe Simulated Kafka Feedback Stream

In production, these events would come from Kafka. In Colab demo mode, an in-memory queue simulates the same producer-consumer pattern without requiring a Kafka broker.


In [ ]:
import queue
import random
from datetime import datetime

random.seed(42)

rating_event_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("recipe_id", IntegerType(), True),
    StructField("rating", FloatType(), True),
    StructField("event_type", StringType(), True),
    StructField("timestamp", StringType(), True)
])

event_queue = queue.Queue()


def produce_rating_event(user_id, recipe_id, rating):
    event = {
        "user_id": int(user_id),
        "recipe_id": int(recipe_id),
        "rating": float(rating),
        "event_type": "rating",
        "timestamp": datetime.utcnow().isoformat()
    }
    event_queue.put(json.dumps(event))
    print("Produced event:", event)
    return event


def consume_events(max_events=10):
    events = []
    while not event_queue.empty() and len(events) < max_events:
        event = json.loads(event_queue.get())
        print("Consumed event:", event)
        events.append(event)
    return events


sample_recipe_ids = [row["recipe_id"] for row in final_ratings.select("recipe_id").distinct().limit(10).collect()]
sample_user_ids = [row["user_id"] for row in final_ratings.select("user_id").distinct().limit(5).collect()]

for _ in range(10):
    produce_rating_event(
        random.choice(sample_user_ids),
        random.choice(sample_recipe_ids),
        random.randint(1, 5)
    )

consumed_events = consume_events()
streaming_events_consumed = len(consumed_events)

if consumed_events:
    streaming_events_df = spark.createDataFrame(consumed_events, schema=rating_event_schema)
else:
    streaming_events_df = spark.createDataFrame([], schema=rating_event_schema)

print("Streaming events consumed:", streaming_events_consumed)
streaming_events_df.show(truncate=False)


### B. Optional Real Kafka Structured Streaming

This block is intentionally disabled by default. It should only be used in cluster mode when Kafka is available and the Spark session includes the Kafka connector package.


In [ ]:
kafka_query = None

if USE_REAL_KAFKA and RUN_MODE == "cluster":
    try:
        from kafka import KafkaProducer
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kafka-python"])
        from kafka import KafkaProducer

    kafka_raw_df = spark.readStream                 .format("kafka")                 .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)                 .option("subscribe", KAFKA_TOPIC)                 .option("startingOffsets", "latest")                 .load()

    kafka_events_df = kafka_raw_df.selectExpr("CAST(value AS STRING) as json_str")                 .select(F.from_json(F.col("json_str"), rating_event_schema).alias("data"))                 .select("data.*")

    kafka_query = kafka_events_df.writeStream                 .format("memory")                 .queryName("live_recipe_rating_events")                 .outputMode("append")                 .start()

    print("Started real Kafka stream.")

    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
        value_serializer=lambda value: json.dumps(value).encode("utf-8")
    )

    test_event = {
        "user_id": int(sample_user_ids[0]),
        "recipe_id": int(sample_recipe_ids[0]),
        "rating": 5.0,
        "event_type": "rating",
        "timestamp": datetime.utcnow().isoformat()
    }

    producer.send(KAFKA_TOPIC, test_event)
    producer.flush()
    producer.close()
    print("Produced Kafka event:", test_event)
else:
    print("USE_REAL_KAFKA=False or RUN_MODE is not 'cluster'; skipping real Kafka stream.")


## Incremental Feedback and Scheduled Retraining

In a production pipeline, new Kafka events would be appended to the training store and ALS would be retrained on a schedule. For demo safety, this notebook shows ingestion and append logic without launching another expensive training job.


In [ ]:
updated_ratings = final_ratings
new_feedback_count = 0
updated_ratings_count = filtered_ratings_count

if consumed_events:
    new_feedback_df = streaming_events_df.select("user_id", "recipe_id", "rating")
    updated_ratings = final_ratings.unionByName(new_feedback_df)
    new_feedback_count = new_feedback_df.count()
    updated_ratings_count = updated_ratings.count()

    print("Original ratings count:", f"{filtered_ratings_count:,}")
    print("New feedback count:", f"{new_feedback_count:,}")
    print("Updated ratings count:", f"{updated_ratings_count:,}")
else:
    print("No feedback events were consumed, so the ratings table was unchanged.")


## Results Summary

The table below collects the final project outputs that are most useful during the demo: dataset scale, filtered training size, model quality, generated recommendations, storage paths, and streaming ingestion counts.


In [ ]:
summary_rows = [
    ("Raw recipe rows", f"{raw_recipe_count:,}"),
    ("Raw interaction rows", f"{raw_interaction_count:,}"),
    ("Recipes after cleaning", f"{clean_recipe_count:,}"),
    ("Valid interactions for analysis", f"{num_interactions:,}"),
    ("Filtered ratings rows", f"{filtered_ratings_count:,}"),
    ("Filtered unique users", f"{filtered_user_count:,}"),
    ("Filtered unique recipes", f"{filtered_recipe_count:,}"),
    ("User-recipe matrix sparsity", f"{matrix_sparsity:.6%}"),
    ("ALS rank", str(als_rank)),
    ("RMSE", f"{rmse:.4f}"),
    ("MAE", f"{mae:.4f}"),
    ("Top-N recommendations generated", f"{topn_recommendations_generated:,}"),
    ("Model save path", model_save_path),
    ("Processed ratings save path", processed_ratings_path),
    ("Processed recipe features save path", processed_recipes_path),
    ("Streaming events consumed", str(streaming_events_consumed)),
]

summary_df = pd.DataFrame(summary_rows, columns=["Metric", "Value"])
display(summary_df)


## Big Data Techniques Used

- [x] Large Food.com dataset with 1M+ interactions
- [x] PySpark distributed DataFrame processing
- [x] Spark SQL aggregations
- [x] Spark-based feature engineering
- [x] Parquet storage
- [x] Optional HDFS storage in cluster mode
- [x] Spark MLlib ALS collaborative filtering
- [x] 50-dimensional latent vectors
- [x] RMSE and MAE evaluation
- [x] Top-N recommendation generation
- [x] Kafka-style streaming feedback simulation
- [x] Optional real Kafka Structured Streaming
- [x] Model persistence


## Changes from Proposal

The original proposal included Apache Kafka for streaming and HDFS for distributed storage. The final notebook supports these through a cluster mode. For the live Colab demo, the notebook uses a reliable simulation of Kafka events and local or Drive storage because Colab is not a persistent Hadoop cluster. The core Big Data components are preserved: PySpark preprocessing, Spark DataFrame transformations, Spark MLlib ALS training, Parquet storage, large-scale Food.com data, recommendation generation, and model evaluation.


## README

**How to run in Colab**

1. Keep `RUN_MODE="colab"` and `USE_REAL_KAFKA=False`.
2. Upload `RAW_recipes.csv` and `RAW_interactions.csv` into `LOCAL_DATA_DIR`, or provide Kaggle credentials through environment variables or `kaggle.json`.
3. Run the notebook from top to bottom.
4. Outputs are written to `LOCAL_OUTPUT_DIR`. If you want Google Drive storage, mount Drive manually and change `LOCAL_OUTPUT_DIR` to a Drive path before running the storage cells.

**How to run in cluster mode with HDFS and Kafka**

1. Set `RUN_MODE="cluster"`.
2. Stage the raw CSV files in `HDFS_RAW_DIR`.
3. Set `USE_REAL_KAFKA=True` only when Kafka is available and the Spark runtime can load the Kafka connector package.
4. Processed Parquet data, metrics, and the ALS model will be saved under the configured HDFS paths.

**Required dataset files**

- `RAW_recipes.csv`
- `RAW_interactions.csv`

**Output locations**

- Local demo outputs: `LOCAL_OUTPUT_DIR`
- Cluster processed data: `HDFS_PROCESSED_DIR`
- Cluster model path: `HDFS_MODEL_DIR`
- Cluster metrics or streaming outputs: `HDFS_OUTPUT_DIR`


In [ ]:
# Optional cleanup for repeated demo runs.
# Uncomment any lines you want to use after the presentation.

# import shutil
# shutil.rmtree(LOCAL_OUTPUT_DIR, ignore_errors=True)
# spark.catalog.clearCache()
# spark.stop()
